# Red Bullseye YOLOv8 Training — Combined Dataset v2

**Dataset:** 3,299 images combined (indoor + outdoor)
- 911 indoor images (47 originals × ~15 augmentations)
- 2,388 outdoor NAVO images (145 labeled + 68 background + augmentations)
- 68 background images (outdoor scenes with no bullseye — teaches robustness)

**Class:** `bullseye` (class 0)
**Output:** `best.pt` → deploy on Jetson with `X_5_bullseye.py`

In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

In [ ]:
!git clone https://github.com/niyatikapadia/bullseye-landing-dataset.git
import os
n_img = len(os.listdir('bullseye-landing-dataset/combined_dataset/images/train'))
n_lbl = len(os.listdir('bullseye-landing-dataset/combined_dataset/labels/train'))
print(f'Images: {n_img}  Labels: {n_lbl}')

In [ ]:
import yaml
from pathlib import Path
dataset_path = Path('/content/bullseye-landing-dataset/combined_dataset').resolve()
yaml_path = dataset_path / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({'path': str(dataset_path), 'train': 'images/train',
               'val': 'images/train', 'nc': 1, 'names': ['bullseye']}, f)
print(open(yaml_path).read())

In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data      = str(yaml_path),
    epochs    = 150,
    imgsz     = 640,
    batch     = 16,
    device    = 0,
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs = 5,
    hsv_h     = 0.015,
    hsv_s     = 0.4,
    hsv_v     = 0.4,
    degrees   = 10.0,
    translate = 0.1,
    scale     = 0.4,
    flipud    = 0.3,
    fliplr    = 0.5,
    mosaic    = 0.5,
    patience  = 30,
    save_period = 25,
    project   = 'bullseye',
    name      = 'v2_combined',
    exist_ok  = True,
    verbose   = True,
)
print('Training complete!')

In [ ]:
import glob
from IPython.display import Image, display
results_files = glob.glob('/content/**/results.png', recursive=True)
if results_files:
    display(Image(results_files[0], width=900))

In [ ]:
import glob
weights = glob.glob('/content/**/best.pt', recursive=True)
print('Found:', weights)
best_model = YOLO(weights[0])
metrics = best_model.val(data=str(yaml_path), imgsz=640, verbose=True)
print(f'\nmAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'Precision:{metrics.box.mp:.4f}')
print(f'Recall:   {metrics.box.mr:.4f}')

In [ ]:
import glob, random, cv2, matplotlib.pyplot as plt
from pathlib import Path
test_images = random.sample(
    glob.glob('bullseye-landing-dataset/navo_original_images/*.jpg'), 6)
fig, axes = plt.subplots(2,3,figsize=(18,8))
for ax, img_path in zip(axes.flat, test_images):
    img = cv2.imread(img_path)
    res = best_model(img, imgsz=640, conf=0.3, verbose=False)
    for r in res:
        for box in r.boxes:
            x1,y1,x2,y2 = map(int,box.xyxy[0])
            cx,cy = (x1+x2)//2,(y1+y2)//2
            cv2.rectangle(img,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.drawMarker(img,(cx,cy),(0,0,255),cv2.MARKER_CROSS,30,2)
            cv2.putText(img,f'{float(box.conf[0]):.2f}',(x1,y1-8),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
    ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
    ax.set_title(Path(img_path).name,fontsize=8)
    ax.axis('off')
plt.suptitle('Inference on Outdoor NAVO Images',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
import glob
from google.colab import files
weights = glob.glob('/content/**/best.pt', recursive=True)
files.download(weights[0])
print('Downloaded: best.pt')
print('Deploy: python3 X_5_bullseye.py --weights best.pt --headless --conf 0.6')